## Step 1: Import Libraries and Keys

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display
import json

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None: 
    raise Exception("OpenAI API Key is not set!")

client = OpenAI(api_key=OPENAI_API_KEY)



## Step 2: Setup Pushover
 
 - Setup account in the browser: done!
 - Setup app on phone, login to same account: done!
 - In browser, create Application/API Token: done!
 - Copy tokens into the .env file

 

In [2]:
load_dotenv()

True

In [3]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

In [4]:
#Test Pusheover
import requests

def send_notification(message: str):
    payload = {'user': pushover_user, 'token': pushover_token, 'device': 'oneplusnordn2005g', 'message': message}
    requests.post(pushover_url, data=payload)


In [5]:
send_notification("Hola Barbara, you are the best!")

## Step 3: Describe Pushover as an LLM tool


In [6]:
send_notification_function = {
    'name': 'send_notification',
    'description': """
                Sends a message to send as a push notification to the users phone via Pushover. 
                User this to alert the user about a new message.""",
    'parameters': {
        'type': 'object',
        'properties': {
            'message': {
                'type': 'string',
                'description': 'The notification message to send to the users device'
            }
        },
        "required": ["message"]
    }
}

## Step 4: Add Pushover to the list of tools for the LLM

In [7]:
tools = [{"type":"function", "function": send_notification_function}]

## Step 5: Calling the tools from an LLM

In [8]:
client = OpenAI()

response = client.chat.completions.create(
    model = 'gpt-4.1-mini',
    messages = [
        {"role": "user", "content": """
                    Please send me a notification that tells me to believe in myself because I am amazing and 
                    tell me why in a few words"""}
    ],
    tools=tools,
    tool_choice="auto" #"required", can pass a dictionary, "auto" is default setting
)

#Check if model wants to call a tool
message = response.choices[0].message

In [9]:
#Check if model wants to call a tool
print(message)


ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_g47xJmb5uVUaUm2hByDdAjYS', function=Function(arguments='{"message":"Believe in yourself because you are resilient, unique, and capable of achieving great things."}', name='send_notification'), type='function')])


In [13]:
if message.tool_calls:
    tool_call = message.tool_calls[0]
    
    args = json.loads(tool_call.function.arguments)

    print(f"The key 'message' in the 'args' variable has value: '{args['message']}' and type {type(args['message'])}")

    #Actually send the notification
    send_notification_function(args['message'])
    print(f"Sent notification: {args['message']}")
else: 
    print(response.choices[0].message)

    

The key 'message' in the 'args' variable has value: 'Believe in yourself because you are resilient, unique, and capable of achieving great things.' and type <class 'str'>


TypeError: 'dict' object is not callable

'Believe in yourself because you are resilient, unique, and capable of achieving great things.'